# HOW TO GLIMT

In [1]:
%load_ext autoreload
%autoreload 2

## Load IFPs

In [3]:
from list_active_ifps import list_active_ifps
ifps = list_active_ifps()

from save_ifps_to_disk import save_ifps_to_disk
id_to_ifp = save_ifps_to_disk(ifps)

from ifps_to_df import ifps_to_df
ifps_to_df(ifps).sort_values(by='id').head(3)

ifp = ifps[0]

200


## Pull top 10 Wikipedia pages by semantic distance from IFP title+description

In [4]:
from wiki_semantic_search import wiki_semantic_search

loading massive wiki index 2025-07-12 20:59:24.264955
loading wiki article titles 2025-07-12 21:00:28.173901
loading sentence transformer model 2025-07-12 21:00:31.793591
done 2025-07-12 21:00:34.069036


In [5]:
title_plus_criteria = f"{ifp['props']['title']} {ifp['props']['details']}".replace('<p><b>', '').replace('</b><p>', '')

In [6]:
wiki_articles = wiki_semantic_search(title_plus_criteria)

In [7]:
wiki_articles[0]

(0.7543423,
 '2008 Russian financial crisis',
 2008 Russian financial crisis (lang: en, variant: None, id: ??, ns: 0))

In [8]:
for score, title, article in wiki_articles:
    print(title)
    print(article.summary[0:100])
    print('===================================================')

2008 Russian financial crisis
The Great Recession in Russia was a crisis during 2008–2009 in the Russian financial markets as well
Russian invasion of Finland (disambiguation)
The Russian invasion of Finland may refer to:

Russo-Swedish War (1495–1497)
Finnish War (1808–1809)
Second Russian Division
The Russian Second League (Russian: Первенство России II дивизиона ФНЛ), formerly the Russian Profes
Georgia and the European Union
The former European Community and Georgia established relations in 1992. After the Maastricht Treaty
Russian atomic bomb
The Soviet atomic bomb project was authorized by Joseph Stalin in the Soviet Union to develop nuclea
2022 Russia-European Union gas dispute
The Russia–EU gas dispute flared up in March 2022 following the invasion of Ukraine on 24 February 2
Russia's invasion of Ukraine
On 24 February 2022, Russia invaded Ukraine, starting the largest and deadliest war in Europe since 
Russia's 102nd Military Base
The Russian 102nd Military Base (Armenian: Ռու

## Gather news for IFPs

In [9]:
from gather_news_for_ifps import gather_news_for_ifps
news = gather_news_for_ifps(ifps)

## Forecast one IFP

### Split news into text and URLs

In [10]:
ifp_id = ifp['id']
ifp_id

458

In [11]:
ifp_news = news[ifp_id]

In [12]:
s1 = ifp_news.split('\n\n')

In [13]:
s2 = [x for x in s1 if '\nSource' in x]

In [14]:
s3 = [x.split('\nSource:') for x in s2]

In [15]:
ifp_news_text = [x[0].split('\nOriginal language:')[0] for x in s3]

In [16]:
ifp_news_sources = [x.split('](')[1][0:-1] for x in [x[1] for x in s3]]

In [17]:
ifp_news = dict([(x,y) for x,y in zip(ifp_news_sources, ifp_news_text)])

In [18]:
ifp_news_sources = [x for x,y in ifp_news.items()]
ifp_news_text = [y for x,y in ifp_news.items()]

In [19]:
len(ifp_news_text)

20

In [20]:
len(wiki_articles)

10

In [21]:
wiki_articles[0]

(0.7543423,
 '2008 Russian financial crisis',
 Great Recession in Russia (lang: en, variant: None, id: 19358251, ns: 0))

In [22]:
combined = list(zip(ifp_news_sources, [x.replace('**', '') for x in ifp_news_text])) + [(z, z.text) for x,y,z in wiki_articles]

### Extract relevant ideas in article for topic

In [96]:
from create_source_summaries import create_source_summaries
source_summaries, sources = \
    create_source_summaries(ifp_id, title_plus_criteria, combined,
            wiki_articles, ifp_news_sources, ifp_wiki_sources)

### Rephrase binary outcomes

In [57]:
from rephrase_binary_outcomes import rephrase_binary_outcomes
rephrase_binary_outcomes(ifp)

## Collect source summaries into block

In [58]:
research = []
for i, summary in enumerate(source_summaries):
    item = f"""```research_summary_{i}
{summary}
```"""
    research.append(item)
research = '\n'.join(research)

## Prompt with rationale fields split into for and against

In [59]:
import os
os.makedirs('glimt/prompt', exist_ok=True)

In [60]:
details = ifp['props']['details']
title = ifp['props']['title']
title

'Will Russia invade Georgia before the end of 2025?'

In [61]:
bins = [x['props']['title'] for x in ifp['bins']]

In [62]:
sb1 = '\n'.join([f"""* O{i+1}. {bin}""" for i, bin in enumerate(bins)])

In [63]:
psum = '+'.join([f'P{i+1}' for i, bin in enumerate(bins)])

In [64]:
pcom = ','.join([f'P{i+1}' for i, bin in enumerate(bins)])

In [65]:
sbins = f"""The question has one of {len(bins)} outcomes namely  

{sb1}

Each outcome Oi has a probability Pi where 0 <= Pi <= 1.
We must have that {psum} = 1.0.
Add some reasonable amount of randomness/noise to the estimation of the branches.
The output is a Python list wrapped by a binProbs tag, in this format:
```binProbs
[{pcom}]
```
"""

print(sbins)

The question has one of 2 outcomes namely  

* O1. Russia will invade Georgia before the end of 2025.
* O2. Russia will not invade Georgia before the end of 2025.

Each outcome Oi has a probability Pi where 0 <= Pi <= 1.
We must have that P1+P2 = 1.0.
Add some reasonable amount of randomness/noise to the estimation of the branches.
The output is a Python list wrapped by a binProbs tag, in this format:
```binProbs
[P1,P2]
```



In [66]:
prompt = f"""
You are a talented, experienced and confident superforecaster. You are asked a question:

```question
{title}
```

You are given details on how to interpret the terms of the question:

```details
{details}
```

Your assistant has research related news and Wikipedia articles and prepared summaries of each one.
Use the data in these research summaries to analyse the question:

{research}

For you to be marked Successful, you must output 3 things:

1. Probabilities for the outcomes of the question.  
{sbins}

2. Reasons your probabilities might be right, wrapped in tag in this format:
```rRight
...reasons you might be right
```

3. Reasons your probabilities might be wrong, wrapped in tag in this format:
```rWrong
...reasons you might be wrong
```
"""

In [67]:
dlgdir = f'glimt/prompt/{ifp_id}'
os.makedirs(dlgdir, exist_ok=True)
fn = f'{dlgdir}/question.txt'
with open(fn, 'w') as f:
    f.write(prompt)

## Run the prompt 5 times

In [68]:
answers = [humor_me(prompt, i+1) for i in range(5)]

START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.3949967940648397

```binProbs
[0.3, 0.7]
```

```rRight
1. **Current Geopolitical Context**: The text highlights that Russia has a significant military presence in Armenia, which could be leveraged for operations in Georgia. The 102nd Military Base in Gyumri is well-equipped with air, ground, and electronic warfare capabilities, suggesting Russia has the means to intervene if necessary.
2. **Historical Precedent**: The Winter War demonstrated Russia's willingness to use military force to secure its borders and influence. This historical context suggests that Russia might be inclined to act similarly in Georgia, especially given the strategic importance of the region.
3. **Armenia's Reliance on Russia**: Armenia's membership in the CSTO and its reliance on Russia for security against threats from Turkey and Azerbaijan could prompt Russia to intervene in Georgia to maintain its

## Extract binProbs, right and wrong

In [69]:
def get_bin_probs(r):
    return eval(r.split('```binProbs')[1].split('```')[0].strip())

In [72]:
binProbs = [get_bin_probs(a) for a in answers]
binProbs

[[0.3, 0.7], [0.3, 0.7], [0.3, 0.7], [0.3, 0.7], [0.3, 0.7]]

In [73]:
def get_rights(r):
    return r.split('```rRight')[1].split('```')[0].strip()

In [76]:
def get_wrongs(r):
    return r.split('```rWrong')[1].split('```')[0].strip()

In [74]:
rights = [get_rights(a) for a in answers]

In [77]:
wrongs = [get_wrongs(a) for a in answers]

## Median forecasts and rationales

In [83]:
import numpy as np
def median_forecast(Fs):
    M = np.array(Fs)
    return np.median(M, axis=0).tolist()

In [84]:
forecast = median_forecast(binProbs)

In [85]:
forecast

[0.3, 0.7]

In [93]:
def median_rationale(Rs):

    WRs = [f"""```forecast
{x}
```""" for x in Rs]
    
    WRS = '\n'.join(WRs)
    
    prompt = f"""
Summarize the gist of the rationale or thinking of the following answers: 

{WRS}

Do not refer to the provided forecasts.  Just give the summary as if it was a new forecast written by you.

THIS SUMMARY MUST BE 1200 CHARACTERS OR LESS.

DO NOT REFER TO THE FORECASTED PROBABILITY, JUST SUMMARIZE REASONING WITHOUT STATING THE CONCLUSION.
"""
    
    medrat = humor_me(prompt)
    
    return medrat

In [94]:
right = median_rationale(rights)

START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.10777047475179037

The assessment of Russia's potential military intervention in Georgia before the end of 2025 considers several key factors. Geopolitically, while Russia maintains a significant military presence in Armenia, its current focus on the war in Ukraine and the resulting economic sanctions reduce the likelihood of opening a new front in Georgia. Historically, Russia has demonstrated willingness to use military force to secure its interests, but the strategic importance of Georgia may not justify the logistical and political effort required for intervention. Armenia's reliance on Russia for security against threats from Turkey and Azerbaijan could influence Russia's decisions, but the strain on its military resources from the Ukraine conflict limits its capacity for additional conflicts. Diplomatic relations and the potential for international backlash, particularl

In [95]:
wrong = median_rationale(wrongs)

START model mistral-small3.2:24b-instruct-2506-q4_K_M
model mistral-small3.2:24b-instruct-2506-q4_K_M minutes 0.10195267995198568

The rationale for assessing the likelihood of a Russian invasion of Georgia revolves around several key factors. The ongoing war in Ukraine has significantly strained Russia's military and economic resources, potentially deterring it from opening a new front. International pressure, including sanctions and military aid to Georgia, could further discourage Russian aggression. However, if the Ukraine conflict escalates or Russia faces setbacks, it might seek to divert attention or secure strategic advantages by invading Georgia. Unpredictable provocations or shifts in Russian leadership could also escalate tensions. Additionally, Russia's history of sudden military actions suggests its intentions may change rapidly. The assumption that Western deterrence will prevent aggression might be overly optimistic, as Russia has acted despite international opposition. 

## Format forecast upload with all the goodies

In [100]:
def jsx_forecast(id: int, 
                 probas: list[float],
                 reason: str,
                 wwcym: str,
                 urls: list[str]):
    reason = reason[0:1200] # hard size on GUI
    wwcym = wwcym[0:1200] # hard size on GUI
    U = repr([{"value": url, "index":i+1,"order":i+1} for i, url in enumerate(urls)]).replace("'", '"')
    return f"""[["ifps","submitFcst",{{"ifpId":{id},"data":{{"probas": {probas} }},"rationale":{{"type":"reason","reason":"{reason}","wwcym":"{wwcym}","urls":{U}}},"publish":true}}]]"""

In [105]:
jsx = jsx_forecast(ifp_id,forecast,right,wrong,sources)

In [106]:
from jsx_request import jsx_request
jsx_request(jsx)

200


[{'probas': [0.3, 0.7],
  'ts': 1752374020251,
  'day': 20282,
  'confDay': 20282,
  'confCnt': 0,
  'isStop': False,
  'rationale': {'type': 'reason',
   'reason': "\x01en:The assessment of Russia's potential military intervention in Georgia before the end of 2025 considers several key factors. Geopolitically, while Russia maintains a significant military presence in Armenia, its current focus on the war in Ukraine and the resulting economic sanctions reduce the likelihood of opening a new front in Georgia. Historically, Russia has demonstrated willingness to use military force to secure its interests, but the strategic importance of Georgia may not justify the logistical and political effort required for intervention. Armenia's reliance on Russia for security against threats from Turkey and Azerbaijan could influence Russia's decisions, but the strain on its military resources from the Ukraine conflict limits its capacity for additional conflicts. Diplomatic relations and the potenti